# 02: FedProx and SCAFFOLD vs FedAvg

Runs FedProx (3 values of mu) and SCAFFOLD on BloodMNIST at alpha 0.1 and 0.5, seeds 0-2, with the same settings as notebook 01 (10 sites, 30 rounds, 2 local epochs, lr 0.02, momentum 0.9). FedAvg and central results from notebook 01 are in `results/step4_bloodmnist_3seeds.json`.

Turn on a T4 GPU (Runtime -> Change runtime type) before running. About 15 minutes.

In [ ]:
!nvidia-smi -L
!rm -rf /content/fedsurveil && git clone -q https://github.com/Nathaniel47/fedsurveil /content/fedsurveil
%cd /content/fedsurveil
!pip install -q -r requirements-ml.txt 2>&1 | tail -1
!python -m fedsurveil.data.fetch
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.__version__, torch.cuda.is_available())

In [ ]:
import subprocess, sys
COMMON = ["--dataset", "bloodmnist", "--rounds", "30", "--lr", "0.02", "--momentum", "0.9", "--local-epochs", "2"]
for seed in [0, 1, 2]:
    for alpha in ["0.1", "0.5"]:
        for extra in (["--algorithm", "scaffold"], *[["--algorithm", "fedprox", "--prox-mu", mu] for mu in ["0.01", "0.1", "1.0"]]):
            cmd = [sys.executable, "-m", "fedsurveil.fl.simulate", *COMMON, "--alpha", alpha, "--seed", str(seed), *extra]
            subprocess.run(cmd, check=True, capture_output=True)
            print("done", seed, alpha, extra)

In [ ]:
import glob, json, os
# Compact copy of every run: [round, acc, macro_f1, worst_site_acc, mean_site_acc] per round
for p in sorted(glob.glob("runs/*.jsonl")):
    rows = [json.loads(line) for line in open(p)]
    curve = [[r["round"], round(r["global_acc"], 4), round(r["macro_f1"], 4), round(r["site_acc_spread"]["min"], 4), round(r["site_acc_spread"]["mean"], 4)] for r in rows if r["type"] == "round"]
    print("RUN", os.path.basename(p), json.dumps(curve), rows[0]["device"])